In [11]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('exoplanet-detection-challenge')

print("Path to competition files:", path)

Path to competition files: /Users/rishabhyadav/.cache/kagglehub/competitions/exoplanet-detection-challenge


In [12]:
# All Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [13]:
# Load the training and testingdata

train_data = pd.read_csv(path + '/train.csv')
test_data = pd.read_csv(path + '/test.csv')
train_data.head()

,star_id,spectral_type,stellar_radius_sr,stellar_mass_sm,stellar_teff_k,stellar_log_g,stellar_luminosity,stellar_metallicity,stellar_rot_period_d,stellar_noise_ppm,label,planet_radius_re,rp_rs_ratio,orbital_period_d,semi_major_au,eccentricity,inclination_deg,impact_parameter,transit_depth_ppm,transit_duration_hr,n_transits_observed,orbital_velocity_kms,transit_snr,planet_eq_temp_k,flux_variability_index,log_period,log_snr
0,STR-000436,F,1.327806,1.398391,6912.100561,4.323038,3.612614,-0.178,21.07,250.66,0,NaN,NaN,NaN,NaN,NaN,NaN,2.183038,22.901432,0.155303,0.0,NaN,0.730875,NaN,0.0914,0.0000,0.5486
1,STR-007250,K,0.912461,0.708236,4533.876776,4.352924,0.298579,-0.067,16.43,236.22,0,NaN,NaN,34.83397,NaN,NaN,NaN,1.326354,2510.133991,13.584701,0.0,NaN,4.513105,NaN,10.6263,3.5789,1.7071
2,STR-005589,K,0.684165,0.716614,4466.620216,4.652811,0.146280,0.083,39.55,162.24,1,5.237,0.07174,13.44170,0.089052,0.4224,89.023,0.539900,5076.490629,2.646656,56.0,79.732,235.808062,503.8,31.2900,2.6701,5.4672
3,STR-011770,B,4.403350,9.499783,22146.527056,4.042372,4379.887860,-0.056,15.55,615.27,0,NaN,NaN,NaN,NaN,NaN,NaN,1.326998,20.652450,0.022175,0.0,NaN,1.261956,NaN,0.0336,0.0000,0.8162
4,STR-003949,G,0.927414,0.974574,5647.596422,4.606604,0.753481,0.194,9.44,268.57,1,1.773,0.01801,7.81420,0.076281,0.0160,88.545,0.479000,334.291352,NaN,46.0,110.276,7.598188,839.8,1.2447,2.1764,2.1516


In [14]:
train_data.set_index('star_id', inplace=True)
test_data.set_index('star_id', inplace=True)

In [ ]:
from sklearn.model_selection import train_test_split

train_y = train_data['label']
train_X = train_data.drop(columns=['label'])
test_X = test_data.copy()

train_X_split , Validate_X , train_y_split, Validate_y = train_test_split(train_X, train_y, test_size=0.3, random_state=67)


In [68]:
# Feature Imputation (impute the physics-based feature)

for df in [train_X_split, Validate_X, test_X]:

    df['rp_rs_ratio'] = df['rp_rs_ratio'].fillna(np.sqrt( df['transit_depth_ppm'] / 1e6) )

    df['planet_radius_re'] = df['planet_radius_re'].fillna(df['rp_rs_ratio'] * df['stellar_radius_sr'] * 109.2)

    df['orbital_period_d'] = df['orbital_period_d'].fillna(df['log_period'].apply(lambda x: 10 ** x))

    df['semi_major_au'] = df['semi_major_au'].fillna( np.cbrt( np.square(df['orbital_period_d'] / 365.25) * (df['stellar_mass_sm'])) )

    df['planet_eq_temp_k'] = df['planet_eq_temp_k'].fillna( df['stellar_teff_k'] * np.sqrt(df['stellar_radius_sr'] / (2 * df['semi_major_au'])) )

    df['orbital_velocity_kms'] = df['orbital_velocity_kms'].fillna( (2*np.pi*df['semi_major_au']*1.496e8) / ((df['orbital_period_d']*24*3600)/1e5) )

str_cols = [c for c in train_X.columns if train_X[c].dtype == 'str']
num_cols = [c for c in train_X.columns if c not in str_cols]
num_cols_to_impute = [c for c in num_cols if train_X[c].isnull().sum() > 0]


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import  OneHotEncoder

# Preprocessing for numerical data
numerical_transformer = SimpleImputer(strategy='median')

train_X_split[num_cols_to_impute] = numerical_transformer.fit_transform(train_X_split[num_cols_to_impute])
Validate_X[num_cols_to_impute] = numerical_transformer.transform(Validate_X[num_cols_to_impute])
test_X[num_cols_to_impute] = numerical_transformer.transform(test_X[num_cols_to_impute])

# Preprocessing for categorical data
categoraical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

categoraical_transformer_cols_train = pd.DataFrame(categoraical_transformer.fit_transform(train_X_split[str_cols]))
categoraical_transformer_cols_test = pd.DataFrame(categoraical_transformer.transform(test_X[str_cols]))
categoraical_transformer_cols_validate = pd.DataFrame(categoraical_transformer.transform(Validate_X[str_cols]))


In [76]:
# One-hot encoding removed index; put it back
categoraical_transformer_cols_train.index = train_X_split.index
categoraical_transformer_cols_validate.index = Validate_X.index
categoraical_transformer_cols_test.index = test_X.index

# Remove categorical columns (will replace with one-hot encoding)
num_X_train = train_X_split.drop(str_cols, axis=1)
num_X_validate = Validate_X.drop(str_cols, axis=1)
num_X_test = test_X.drop(str_cols, axis=1)

# Add one-hot encoded columns to numerical features
categoraical_transformed_train_X = pd.concat([num_X_train, categoraical_transformer_cols_train], axis=1)
categoraical_transformed_Validate_X = pd.concat([num_X_validate, categoraical_transformer_cols_validate], axis=1)
categoraical_transformed_test_X = pd.concat([num_X_test, categoraical_transformer_cols_test], axis=1)

# Ensure all columns have string type
categoraical_transformed_train_X.columns = categoraical_transformed_train_X.columns.astype(str)
categoraical_transformed_Validate_X.columns = categoraical_transformed_Validate_X.columns.astype(str)
categoraical_transformed_test_X.columns = categoraical_transformed_test_X.columns.astype(str)

In [77]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from catboost import CatBoostClassifier

models = {
    'Random_Forest': RandomForestClassifier(random_state=67, class_weight='balanced'),
    'Logistic_Regression' : LogisticRegression(max_iter=1000, random_state=67, class_weight='balanced'),
    'SVC' : SVC(probability=True, random_state=67, class_weight='balanced'),
    'CatBoost_Classifier' : CatBoostClassifier(iterations=1000, random_state=67, learning_rate=0.05, depth=6, verbose=False)
    }

In [84]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

metrics_evaluation_result = [] 


for model_name, model in models.items():
    model.fit(categoraical_transformed_train_X, train_y_split)
    
    pred = model.predict(categoraical_transformed_Validate_X) 
    pred_prob = model.predict_proba(categoraical_transformed_Validate_X)[:, 1]


    metrics_evaluation_result.append({
        "Model": model_name,
        "Accuracy": accuracy_score(Validate_y, pred),
        "Precision": precision_score(Validate_y, pred),
        "Recall": recall_score(Validate_y, pred),
        "F1 Score": f1_score(Validate_y, pred),
        "ROC AUC": roc_auc_score(Validate_y, pred_prob)
    })
    


metrics_evaluation = pd.DataFrame(metrics_evaluation_result)
metrics_evaluation

/Users/rishabhyadav/techrelatedfolder/ai&mlShit/exoplanet_detection_challenge/.venv/lib/python3.14/site-packages/sklearn/svm/_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,Random_Forest,1.0,1.0,1.0,1.0,1.0
1,Logistic_Regression,1.0,1.0,1.0,1.0,1.0
2,SVC,1.0,1.0,1.0,1.0,1.0
3,CatBoost_Classifier,1.0,1.0,1.0,1.0,1.0
